# Least and greatest waterfall-equilibrium prices

This notebook reproduces the computational examples in *Asset Value and Securitization under Heterogeneous Beliefs* and adds a two-tranche multiplicity example. It computes only the least equilibrium price $q_{\mathcal T}^{\min}$ and the greatest equilibrium price $q_{\mathcal T}^{\max}$, using the monotone iterations in Proposition 1. It does not search for intermediate equilibria.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repository_url = (
    "https://github.com/oliverpardo1979/"
    "Asset-value-and-securitization-under-heterogeneous-beliefs-anew.git"
)
repository_dir = Path("/content/asset-value-securitization")

if repository_dir.exists():
    subprocess.run(
        ["git", "-C", str(repository_dir), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", repository_url, str(repository_dir)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(repository_dir / "computational_companion" / "requirements.txt"),
    ],
    check=True,
)
os.chdir(repository_dir)
print(f"Repository ready at {repository_dir}")

## Paper examples

The five cases below compare the no-tranching and tranching versions of the motivating example, an additional two-tranche multiplicity case, and the reviewer's multiplicity example.

In [ ]:
import numpy as np
from IPython.display import Markdown, display

from computational_companion.paper_examples import all_examples
from computational_companion import compute_extreme_equilibria


def format_vector(values):
    return "(" + ", ".join(f"{value:.3f}" for value in values) + ")"


rows = []
for example in all_examples():
    result = compute_extreme_equilibria(example.economy)
    rows.append(
        (
            example.name,
            format_vector(result.q_min),
            format_vector(result.q_max),
            "Yes" if result.multiplicity_detected() else "No",
            max(result.residual_min, result.residual_max),
        )
    )

table = [
    r"| Case | $q^{\min}$ | $q^{\max}$ | Multiplicity | Maximum residual |",
    "| --- | --- | --- | --- | --- |",
]
table.extend(
    f"| {name} | {q_min} | {q_max} | {multiple} | {residual:.2e} |"
    for name, q_min, q_max, multiple, residual in rows
)
display(Markdown("\n".join(table)))

## Two-tranche multiplicity within the motivating family

The additional parameterization retains the two theories used in the motivating example: theory $\mathcal A$ assigns probability $1/3$ to every next-period state, while theory $\mathcal B$ assigns probability $9/10$ to the current state and $1/20$ to each other state. Set

$$R=\frac{21}{20},\qquad d=(0,2,3),\qquad \mathcal T=\{[0,51),[51,\infty)\}.$$

The two monotone iterations yield

$$q_{\mathcal T}^{\min}=\left(\frac{28400}{609},\frac{29560}{609},\frac{1500}{29}\right)\approx(46.634,48.539,51.724),$$

$$q_{\mathcal T}^{\max}=\frac{1}{3187}(154560,162783,169521)\approx(48.497,51.077,53.191).$$

At the least equilibrium, the junior tranche pays only in state $h$. At the greatest equilibrium, it pays in states $m$ and $h$. In state $m$, both tranches are priced by theory $\mathcal A$ at the least equilibrium, whereas theory $\mathcal B$ prices the senior tranche and theory $\mathcal A$ prices the junior tranche at the greatest equilibrium. Thus, two tranches are sufficient for multiple equilibrium prices.

## Verification

The regression tests compare the computed extreme equilibria with the values reported in the paper. They also verify directly that the reported intermediate price vector in the reviewer's example is a fixed point, although the solver does not return it.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "computational_companion.test_waterfall_equilibria",
    ],
    cwd=repository_dir,
    check=True,
)